# Setup

In [1]:
#FOR MAC, USE THIS BLOCK

import os, sys

SUMO_HOME  = r'C:\Program Files (x86)\Eclipse\Sumo'
SUMO_HOME  = '/Library/Frameworks/EclipseSUMO.framework/Versions/1.28/EclipseSUMO/share/sumo'
# PROJ_PATH  = '/Library/Frameworks/EclipseSUMO.framework/Versions/1.22.0/EclipseSUMO/framework/EclipseSUMO.framework/Versions/1.22.0/EclipseSUMO/share/proj'

os.environ['SUMO_HOME']  = SUMO_HOME
# os.environ['PROJ_LIB']   = PROJ_PATH
# os.environ['PROJ_DATA']  = PROJ_PATH

sys.path.append(os.path.join(SUMO_HOME, 'tools'))

# Verify all three
checks = {
    'SUMO_HOME':  os.path.exists(SUMO_HOME),
#     'proj.db':    os.path.exists(os.path.join(PROJ_PATH, 'proj.db')),
    'gtfs2pt.py': os.path.exists(f'{SUMO_HOME}/tools/import/gtfs/gtfs2pt.py'),
}
for k, v in checks.items():
    print(f"{k:15s}: {'✅' if v else '❌ NOT FOUND'}")

SUMO_HOME      : ❌ NOT FOUND
gtfs2pt.py     : ❌ NOT FOUND


In [1]:
#FOR WINDOWS, USE THIS BLOCK

import os, sys

# Which SUMO to use. gtfs2pt needs 1.28.0 or newer: 1.27.1 writes duplicate stop ids with --use-gtfs-stopids
# (#18237) and reuses a stale resources/ cache after the network changes (#18291).
#   "installed" the SUMO from the Windows installer (currently 1.28.0); the installer sets SUMO_HOME
#   "nightly"   a nightly build: unzip https://sumo.dlr.de/daily/sumo-win64-git.zip and point NIGHTLY_HOME at it
# Restart the kernel after switching. Falls back to the installed SUMO if the nightly isn't found.
SUMO_VERSION = "installed"
NIGHTLY_HOME = r'C:\Users\dahh2\sumo-nightly\sumo-win64-git'

# Edit the fallback if SUMO is installed elsewhere (check with: where sumo)
SUMO_HOME = os.environ.get('SUMO_HOME', r'C:\Program Files (x86)\SUMO').rstrip('\\/')
if SUMO_VERSION == "nightly":
    if os.path.exists(os.path.join(NIGHTLY_HOME, 'tools')):
        SUMO_HOME = NIGHTLY_HOME
    else:
        print("nightly not found at %s, using the installed SUMO in %s" % (NIGHTLY_HOME, SUMO_HOME))

# Optional: set this to the folder holding "gtfs data" and your .net.xml
# WORK_DIR = r'C:\Users\YOUR_NAME\path\to\DowntownSeattleSUMO\Simulation\GTFS'
# os.chdir(WORK_DIR)

SUMO_TOOLS = os.path.join(SUMO_HOME, 'tools')
os.environ['SUMO_HOME'] = SUMO_HOME
# The SUMO installer sets a machine-wide PYTHONPATH to the release's tools folder, which wins over SUMO_HOME.
# Override it here (the !python gtfs2pt calls inherit it) and put this SUMO's bin first on PATH.
os.environ['PYTHONPATH'] = SUMO_TOOLS
os.environ['PATH'] = os.path.join(SUMO_HOME, 'bin') + os.pathsep + os.environ['PATH']
# same for this kernel: drop any other SUMO tools folder from sys.path
sys.path = [SUMO_TOOLS] + [p for p in sys.path
                           if not (os.path.exists(os.path.join(p, 'sumolib'))
                                   and os.path.normcase(os.path.abspath(p)) != os.path.normcase(SUMO_TOOLS))]
if 'sumolib' in sys.modules and not os.path.normcase(sys.modules['sumolib'].__file__).startswith(os.path.normcase(SUMO_TOOLS)):
    print("sumolib was already imported from another SUMO -- restart the kernel and run this cell first")

# If you see "Cannot find proj.db" errors, uncomment and point at your proj folder:
# PROJ_PATH = os.path.join(SUMO_HOME, 'data', 'proj')
# os.environ['PROJ_LIB'] = PROJ_PATH
# os.environ['PROJ_DATA'] = PROJ_PATH

checks = {
    'SUMO_HOME': os.path.exists(SUMO_HOME),
    'gtfs2pt.py': os.path.exists(os.path.join(SUMO_TOOLS, 'import', 'gtfs', 'gtfs2pt.py')),
}
for k, v in checks.items():
    print(f"{k:15s}: {'✅' if v else '❌ NOT FOUND'}")
print("SUMO_HOME:", SUMO_HOME)
print(os.popen('"%s" --version' % os.path.join(SUMO_HOME, 'bin', 'sumo')).readline().strip())
print("cwd:", os.getcwd())

SUMO_HOME      : ✅
gtfs2pt.py     : ✅
SUMO_HOME: C:\Program Files (x86)\SUMO
Eclipse SUMO sumo 1.28.0
cwd: c:\Users\dahh2\UW Projects\Seattle_Sumo\DowntownSeattleSUMO\Simulation\GTFS


In [2]:
# cd /Users/ziliqu/Dev/SDOT_Worldcup/Sumo_Test/2
# !cd "C:\Users\Soheil99\0 codes\DowntownSeattleSUMO\Simulation\GTFS"
# !cd


os.getcwd()

'c:\\Users\\dahh2\\UW Projects\\Seattle_Sumo\\DowntownSeattleSUMO\\Simulation\\GTFS'

# GTFS

In [3]:
import pandas as pd
import zipfile
import os

def load_gtfs(zip_path):
    data = {}
    with zipfile.ZipFile(zip_path, 'r') as z:
        for file in z.namelist():
            if file.endswith(".txt"):
                data[file] = pd.read_csv(z.open(file))
    return data


### Filtering to major Seattle GTFS Bus Lines

In [ ]:

# ==============================
# CONFIG (EDIT THESE)
# ==============================

INPUT_ZIP = "gtfs data/kcm_google_transit.zip"
OUTPUT_ZIP = "gtfs data/kcm_google_transit_downtown.zip"

# Select important downtown routes 
# KEEP_ROUTES = ["121"]
# TESTED ROUTES 
# KEEP_ROUTES = ["C Line", "D Line", "E Line","G Line", "H Line"]
KEEP_ROUTES = ["4","7","8","36","2","40","70","62","14","1","5","49","13","11","3","150","10","12","124","21","101","132","28","131","27","24","33","125","102","322","256","17","57","56","113"]


# Downtown Seattle bounding box
LAT_MIN, LAT_MAX = 47.58, 47.65
LON_MIN, LON_MAX = -122.37, -122.30

# ==============================
# LOAD GTFS FILES
# ==============================



gtfs = load_gtfs(INPUT_ZIP)

routes = gtfs["routes.txt"]
trips = gtfs["trips.txt"]
stop_times = gtfs["stop_times.txt"]
stops = gtfs["stops.txt"]

# ==============================
# STEP 1: FILTER ROUTES
# ==============================

filtered_routes = routes[routes["route_short_name"].isin(KEEP_ROUTES)]

print(f"Kept routes: {len(filtered_routes)}")

# ==============================
# STEP 2: FILTER TRIPS
# ==============================

filtered_trips = trips[trips["route_id"].isin(filtered_routes["route_id"])]

print(f"Kept trips: {len(filtered_trips)}")

# ==============================
# STEP 3: FILTER STOP TIMES
# ==============================

filtered_stop_times = stop_times[
    stop_times["trip_id"].isin(filtered_trips["trip_id"])
]

# ==============================
# STEP 4: FILTER STOPS (GEOGRAPHIC)
# ==============================

filtered_stops = stops[
    (stops["stop_lat"] >= LAT_MIN) &
    (stops["stop_lat"] <= LAT_MAX) &
    (stops["stop_lon"] >= LON_MIN) &
    (stops["stop_lon"] <= LON_MAX)
]
## TODO ---soheil comment:
# do we have trips where the removed stops are in mid trips? trips that leave simulation area and then come back to it?
print(f"Kept stops in downtown: {len(filtered_stops)}")

# Keep only stop_times that use those stops
filtered_stop_times = filtered_stop_times[
    filtered_stop_times["stop_id"].isin(filtered_stops["stop_id"])
]

# ==============================
# STEP 5: CLEAN TRIPS (remove empty)
# ==============================

valid_trip_ids = filtered_stop_times["trip_id"].unique()
filtered_trips = filtered_trips[filtered_trips["trip_id"].isin(valid_trip_ids)]

# ==============================
# STEP 6: WRITE NEW GTFS ZIP
# ==============================

with zipfile.ZipFile(OUTPUT_ZIP, 'w') as z:
    def write(df, name):
        z.writestr(name, df.to_csv(index=False))

    write(filtered_routes, "routes.txt")
    write(filtered_trips, "trips.txt")
    write(filtered_stop_times, "stop_times.txt")
    write(filtered_stops, "stops.txt")

    # Copy every other untouched file as-is (shapes.txt, calendar.txt, calendar_dates.txt,
    # agency.txt, etc.) instead of a hardcoded allowlist 
    for fname in gtfs:
        if fname not in ['routes.txt', 'trips.txt', 'stop_times.txt', 'stops.txt']:
            write(gtfs[fname], fname)

print(f"✅ Filtered GTFS saved to: {OUTPUT_ZIP}")


Kept routes: 0
Kept trips: 0
Kept stops in downtown: 649
✅ Filtered GTFS saved to: gtfs data/kcm_google_transit_downtown.zip


In [73]:
# Snap the filtered bus stops onto the network before gtfs2pt (run after the filtering cell above)
import io
import math
import zipfile
import xml.etree.ElementTree as ET

import pandas as pd
import sumolib


def snap_gtfs_stops_to_network(gtfs_zip, net_file, patched_stops_file=None, vclass="bus",
                               radius=35.0, max_angle=45.0, end_margin=7.0):
    """Check every stop of a (filtered) GTFS zip against the SUMO network before gtfs2pt.

    gtfs2pt places a stop on whichever route edge is closest to the GTFS coordinate, which
    goes wrong at junctions (closer edge is the cross street -> bus loops around to reach it)
    and at the network boundary (stop beyond the network snaps onto a pseudo edge).
    For each stop, using the direction of travel from shapes.txt:
    - candidate edges: allow `vclass`, within `radius` m, heading within `max_angle` deg of
      the travel direction; pseudo* boundary edges and motorway mainlines don't count
    - candidate found -> move the stop coordinate onto the nearest one (kept `end_margin` m
      away from the edge ends so it can't snap to the neighbouring edge at the junction)
    - no real edge at all within `radius` -> stop is outside the network: removed from
      stop_times, so the trip ends at the previous stop (trips left with < 2 stops are removed)
    - only cross streets nearby (no road of any kind in the travel direction) -> the street is
      not in the network: removed like a stop outside the network
    - a road in the travel direction exists but no bus edge does -> left unchanged with a warning
      (usually a direction or permission problem worth fixing rather than hiding)
    Stops listed in `patched_stops_file` (ids gtfs_<stop_id>) are left to the patch.
    The zip is rewritten in place.
    """
    with zipfile.ZipFile(gtfs_zip) as z:
        raw = {n: z.read(n) for n in z.namelist()}
    read = lambda n: pd.read_csv(io.BytesIO(raw[n]), dtype=str, keep_default_na=False)
    stops, stop_times, trips, shapes = (read(n) for n in
                                        ("stops.txt", "stop_times.txt", "trips.txt", "shapes.txt"))

    net = sumolib.net.readNet(net_file)
    ROAD_CLASSES = ("passenger", "bus", "delivery", "truck")   # any road a vehicle could use
    xy = lambda lat, lon: net.convertLonLat2XY(float(lon), float(lat))
    bearing = lambda a, b: math.degrees(math.atan2(b[0] - a[0], b[1] - a[1])) % 360
    angle_diff = lambda a, b: abs((a - b + 180) % 360 - 180)

    def nearest_segment(points, p):
        """(distance, bearing) of the segment of polyline `points` closest to p"""
        best = None
        for a, b in zip(points, points[1:]):
            if a == b:
                continue
            d = sumolib.geomhelper.distancePointToLine(p, a, b)
            if d < 0:  # p projects outside the segment
                d = min(math.dist(p, a), math.dist(p, b))
            if best is None or d < best[0]:
                best = (d, bearing(a, b))
        return best

    patched = set()
    if patched_stops_file:
        # parse as XML so commented-out pins don't count (gtfs2pt ignores them too)
        root = ET.parse(patched_stops_file).getroot()
        patched = {el.get("id")[len("gtfs_"):] for el in root.iter()
                   if el.tag in ("busStop", "trainStop") and el.get("id", "").startswith("gtfs_")}

    shapes["n"] = shapes.shape_pt_sequence.astype(int)
    shape_xy, shape_dist = {}, {}
    for sid, grp in shapes.groupby("shape_id"):
        grp = grp.sort_values("n")
        shape_xy[sid] = [xy(r.shape_pt_lat, r.shape_pt_lon) for r in grp.itertuples()]
        shape_dist[sid] = pd.to_numeric(grp.get("shape_dist_traveled"), errors="coerce").tolist() \
            if "shape_dist_traveled" in grp else None
    stop_xy = {r.stop_id: xy(r.stop_lat, r.stop_lon) for r in stops.itertuples()}

    def bearing_at(shape_id, d):
        """bearing of the shape segment at distance d along it (shape_dist_traveled)"""
        pts, dist = shape_xy[shape_id], shape_dist[shape_id]
        for i in range(len(pts) - 1):
            if dist[i] <= d <= dist[i + 1] and pts[i] != pts[i + 1]:
                return bearing(pts[i], pts[i + 1])
        return None

    # direction of travel at each stop, from the shape of every trip pattern serving it.
    # Use the stop's distance along the shape where GTFS gives it: the nearest segment is wrong where a
    # route passes the same street twice (loops like the G Line pass Madison & Terry in both directions).
    st = stop_times.merge(trips[["trip_id", "shape_id"]], on="trip_id")
    if "shape_dist_traveled" in st:
        st["d"] = pd.to_numeric(st.shape_dist_traveled, errors="coerce")
    else:
        st["d"] = float("nan")
    directions = {}
    for (shape_id, stop_id, d), _ in st.groupby(["shape_id", "stop_id", "d"], dropna=False):
        if shape_id not in shape_xy:
            continue
        b = None
        if d == d and shape_dist[shape_id] and all(x == x for x in shape_dist[shape_id]):   # d is not NaN
            b = bearing_at(shape_id, d)
        if b is None:
            seg = nearest_segment(shape_xy[shape_id], stop_xy[stop_id])
            b = seg[1] if seg else None
        if b is not None:
            directions.setdefault(stop_id, []).append(b)

    moved, dropped, warned = [], [], []
    for r in stops.itertuples():
        sid = r.stop_id
        if sid not in directions or sid in patched:
            continue
        dirs = directions[sid]
        if max(angle_diff(a, dirs[0]) for a in dirs) > max_angle:
            warned.append((sid, r.stop_name, "served in conflicting directions, left unchanged"))
            continue
        travel = dirs[0]
        p = stop_xy[sid]
        real, candidates, parallel_roads = [], [], []
        for e, dist in net.getNeighboringEdges(p[0], p[1], radius):
            if e.getID().startswith("pseudo"):
                continue
            parallel = angle_diff(nearest_segment(e.getShape(), p)[1], travel) <= max_angle
            if parallel and any(l.allows(c) for l in e.getLanes() for c in ROAD_CLASSES):
                parallel_roads.append(e)
            if e.getType() == "highway.motorway" or not any(l.allows(vclass) for l in e.getLanes()):
                continue
            real.append(e)
            if parallel:
                candidates.append((dist, e))
        if not real:
            dropped.append((sid, r.stop_name))
            continue
        if not candidates and not parallel_roads:
            # only cross streets nearby: the street the bus runs on is not in the network
            dropped.append((sid, r.stop_name + " (street not in the network, only cross streets nearby)"))
            continue
        if not candidates:
            warned.append((sid, r.stop_name, "no %s edge in the direction of travel within %.0f m "
                           "(nearest: %s), left unchanged" % (vclass, radius, real[0].getID())))
            continue
        dist, edge = min(candidates, key=lambda c: c[0])
        lane = min((l for l in edge.getLanes() if l.allows(vclass)),
                   key=lambda l: l.getClosestLanePosAndDist(p)[1])
        pos = lane.getClosestLanePosAndDist(p)[0]
        pos = min(max(pos, min(end_margin, lane.getLength() / 2)), max(lane.getLength() - end_margin, lane.getLength() / 2))
        lon, lat = net.convertXY2LonLat(*sumolib.geomhelper.positionAtShapeOffset(lane.getShape(), pos))
        stops.loc[stops.stop_id == sid, ["stop_lat", "stop_lon"]] = ["%.7f" % lat, "%.7f" % lon]
        moved.append((sid, r.stop_name, edge.getID(), dist))

    if dropped:
        drop_ids = {sid for sid, _ in dropped}
        stop_times = stop_times[~stop_times.stop_id.isin(drop_ids)]
        counts = stop_times.groupby("trip_id").size()
        short = set(counts[counts < 2].index) | (set(trips.trip_id) - set(counts.index))
        stop_times = stop_times[~stop_times.trip_id.isin(short)]
        trips = trips[~trips.trip_id.isin(short)]
    else:
        short = set()

    raw["stops.txt"] = stops.to_csv(index=False).encode("utf-8")
    raw["stop_times.txt"] = stop_times.to_csv(index=False).encode("utf-8")
    raw["trips.txt"] = trips.to_csv(index=False).encode("utf-8")
    with zipfile.ZipFile(gtfs_zip, "w", zipfile.ZIP_DEFLATED) as z:
        for name, data in raw.items():
            z.writestr(name, data)

    print(f"{gtfs_zip}: snapped {len(moved)} stop(s) onto the network, "
          f"dropped {len(dropped)} outside it ({len(short)} trip(s) left with < 2 stops removed), "
          f"{len(warned)} warning(s)")
    for sid, name in dropped:
        print(f"   dropped {sid:>6} {name}")
    for sid, name, msg in warned:
        print(f"   WARNING {sid:>6} {name}: {msg}")
    return moved, dropped, warned


snap_gtfs_stops_to_network("gtfs data/kcm_google_transit_downtown.zip", "soheil_seattle_merged_pedspeed.net.xml", "bus/patched_stops.add.xml");

gtfs data/kcm_google_transit_downtown.zip: snapped 0 stop(s) onto the network, dropped 0 outside it (0 trip(s) left with < 2 stops removed), 0 warning(s)


In [7]:
# busgtfs = load_gtfs('gtfs data/kcm_google_transit_downtown.zip')
# stops = busgtfs['stops.txt']
# stops.stop_name.unique()
# stops.head()

In [8]:
# railgtfs = load_gtfs('gtfs data/rail_gtfs.zip')
# stops = railgtfs['stops.txt']
# # stops.stop_name.unique()
# # stops.head()
# # stops[stops["stop_name"].str.contains("Lynnwood", case=False, na=False)]
# LAT_MIN, LAT_MAX = 47.5798124250899, 47.65143948560587
# LON_MIN, LON_MAX = -122.38544987028668, -122.29961918065126


# filtered_stops = stops[
#     (stops["stop_lat"] >= LAT_MIN) &
#     (stops["stop_lat"] <= LAT_MAX) &
#     (stops["stop_lon"] >= LON_MIN) &
#     (stops["stop_lon"] <= LON_MAX)
# ]

# filtered_stops

In [9]:
# ==============================
# CONFIG (EDIT THESE)
# ==============================

INPUT_ZIP = "gtfs data/rail_gtfs.zip"
OUTPUT_ZIP = "gtfs data/rail_gtfs_downtown.zip"

# Select important downtown routes 
KEEP_ROUTES = [
    "1 Line", "2 Line"
]

# Downtown Seattle bounding box
# #old ones
LAT_MIN, LAT_MAX = 47.31, 47.84
LON_MIN, LON_MAX = -122.43, -122.09

# new ones
# LAT_MIN, LAT_MAX = 47.5798124250899, 47.65143948560587
# LON_MIN, LON_MAX = -122.38544987028668, -122.29961918065126
# 47.5798124250899, -122.32752854299258 (below SODO)
# 47.65143948560587, -122.30397070065698 (above UW)
# 47.59038693450619, -122.29961918065126 (judkins park)
# 47.59525640734678, -122.38544987028668 (west seattle)

# ==============================
# LOAD GTFS FILES
# ==============================

gtfs = load_gtfs(INPUT_ZIP)

routes = gtfs["routes.txt"]
trips = gtfs["trips.txt"]
stop_times = gtfs["stop_times.txt"]
stops = gtfs["stops.txt"]

# ==============================
# STEP 1: FILTER ROUTES
# ==============================

filtered_routes = routes[routes["route_short_name"].isin(KEEP_ROUTES)]

print(f"Kept routes: {len(filtered_routes)}")

# ==============================
# STEP 2: FILTER TRIPS
# ==============================

filtered_trips = trips[trips["route_id"].isin(filtered_routes["route_id"])]

print(f"Kept trips: {len(filtered_trips)}")

# ==============================
# STEP 3: FILTER STOP TIMES
# ==============================

filtered_stop_times = stop_times[
    stop_times["trip_id"].isin(filtered_trips["trip_id"])
]

# ==============================
# STEP 4: FILTER STOPS (GEOGRAPHIC)
# ==============================

filtered_stops = stops[
    (stops["stop_lat"] >= LAT_MIN) &
    (stops["stop_lat"] <= LAT_MAX) &
    (stops["stop_lon"] >= LON_MIN) &
    (stops["stop_lon"] <= LON_MAX)
]

print(f"Kept stops in downtown: {len(filtered_stops)}")

# Keep only stop_times that use those stops
filtered_stop_times = filtered_stop_times[
    filtered_stop_times["stop_id"].isin(filtered_stops["stop_id"])
]

# ==============================
# STEP 5: CLEAN TRIPS (remove empty)
# ==============================

valid_trip_ids = filtered_stop_times["trip_id"].unique()
filtered_trips = filtered_trips[filtered_trips["trip_id"].isin(valid_trip_ids)]

# ==============================
# STEP 6: WRITE NEW GTFS ZIP
# ==============================

with zipfile.ZipFile(OUTPUT_ZIP, 'w') as z:
    def write(df, name):
        z.writestr(name, df.to_csv(index=False))

    write(filtered_routes, "routes.txt")
    write(filtered_trips, "trips.txt")
    write(filtered_stop_times, "stop_times.txt")
    write(filtered_stops, "stops.txt")

    # Copy unchanged files if they exist
#     for fname in ["calendar.txt", "calendar_dates.txt", "agency.txt"]:
#         if fname in gtfs:
#             write(gtfs[fname], fname)
    for fname in gtfs:
        if fname not in ['routes.txt', 'trips.txt', 'stop_times.txt', 'stops.txt']:
            write(gtfs[fname], fname)

print(f"✅ Filtered GTFS saved to: {OUTPUT_ZIP}")


Kept routes: 2
Kept trips: 17803
Kept stops in downtown: 309
✅ Filtered GTFS saved to: gtfs data/rail_gtfs_downtown.zip


### Adding GTFS to Sumo Network

#### suggestion: make sure previous outputs are deleted (fcd and resources folders)

In [10]:
# !python $SUMO_HOME/tools/import/gtfs/gtfs2pt.py \
# -n merged.net.xml \
# --gtfs rail_gtfs_fil.zip \
# --date 20260329 \
# --modes=tram \
# --repair \
# --verbose

# !python "$SUMO_HOME/tools/import/gtfs/gtfs2pt.py" \
# -n merged_v2.net.xml \
# --gtfs "gtfs data/rail_gtfs_downtown.zip" \
# --date 20260329 \
# --modes=tram \
# --repair \
# --verbose

# outputs (routes, stops, vtypes) go to rail/ instead of the GTFS root
os.makedirs("rail", exist_ok=True)

!python "$SUMO_HOME/tools/import/gtfs/gtfs2pt.py" \
-n soheil_seattle_merged_pedspeed.net.xml \
--gtfs "gtfs data/rail_gtfs_downtown.zip" \
--date 20260329 \
--modes=tram \
--route-output rail/gtfs_pt_vehicles.add.xml \
--additional-output rail/gtfs_pt_stops.add.xml \
--vtype-output rail/vtypes.xml \
--repair \
--verbose

Loading net
function import_gtfs called at Thu, 08 Oct 2026 15:20:25 +0000
Loading GTFS data "gtfs data/rail_gtfs_downtown.zip"
function import_gtfs finished after 2.699413 seconds
Success.
Success.
mapping tram
mapping trace with 15 points ... (265 router calls)
mapping trace with 26 points ... (303 router calls)
mapping trace with 22 points ... (274 router calls)
mapping trace with 26 points ... (303 router calls)
mapping trace with 4 points ... (5 router calls)
mapping trace with 8 points ... (9 router calls)
mapping trace with 11 points ... (36 router calls)
mapping trace with 22 points ... (287 router calls)
mapping trace with 11 points ... (36 router calls)
mapping trace with 26 points ... (291 router calls)
mapping trace with 26 points ... (291 router calls)
mapping trace with 1 points ... (0 router calls)
mapping trace with 4 points ... (21 router calls)
mapping trace with 6 points ... (23 router calls)
mapping trace with 10 points ... (33 router calls)
mapping trace with 19 po

In [74]:
# !python "$SUMO_HOME/tools/import/gtfs/gtfs2pt.py" \
# -n soheil_seattle_merged.net.xml \
# --gtfs "gtfs data/kcm_google_transit_downtown.zip" \
# --date 20260617 \
# --modes=bus \
# --repair \
# --verbose

# use this option to avoid duplicate bus stop error -- the second options gives you readability
# outputs (routes, stops, vtypes) go to bus/ instead of the GTFS root
# bus/patched_stops.add.xml pins stops whose GTFS coordinate snaps to the wrong edge
# (e.g. 1559 belongs on Alaskan Way before the turn, but its coordinate is closer to Columbia St)
os.makedirs("bus", exist_ok=True)

!python "$SUMO_HOME/tools/import/gtfs/gtfs2pt.py" \
-n soheil_seattle_merged_pedspeed.net.xml \
--gtfs "gtfs data/kcm_google_transit_downtown.zip" \
--date 20260617 \
--modes=bus \
--route-output bus/gtfs_pt_vehicles.add.xml \
--additional-output bus/gtfs_pt_stops.add.xml \
--vtype-output bus/vtypes.xml \
--patched-stops bus/patched_stops.add.xml \
--use-gtfs-stopids \
--repair \
--verbose

Loading net
function import_gtfs called at Fri, 09 Oct 2026 03:31:37 +0000
Loading GTFS data "gtfs data/kcm_google_transit_downtown.zip"
Warning! No GTFS data found for the given modes bus.
Warning! No GTFS data found for the given date 20260617.
function import_gtfs finished after 0.348477 seconds


Warning! GTFS data did not contain any trips with stops within the given bounding box area.


In [ ]:

with open("tram.add.xml", "w") as f:
    f.write("""<additional>
    <vType id="tram"
           vClass="rail"
           accel="1.2"
           decel="1.5"
           sigma="0.3"
           length="40"
           maxSpeed="25"
           guiShape="rail"/>
</additional>
""")


In [6]:
with open("bus.add.xml", "w") as f:
    f.write("""<additional>
    <vType id="bus"
           vClass="bus"
           accel="1.0"
           decel="4.5"
           sigma="0.5"
           length="12"
           maxSpeed="16.7"
           guiShape="bus"/>
</additional>
""")


In [12]:
import re

with open("bus/gtfs_pt_stops.add.xml") as f:
    content = f.read()

# replace trainStop with busStop
content = content.replace("trainStop", "busStop")

with open("gtfs_pt_stops_fixed.add.xml", "w") as f:
    f.write(content)


In [ ]:
!sumo-gui \
-n merged.net.xml \
-a tram.add.xml,gtfs_pt_stops_fixed.add.xml,gtfs_pt_vehicles.add.xml


In [ ]:
!sumo-gui \
-n merged.net.xml \
-a bus.add.xml,gtfs_pt_stops.add.xml,gtfs_pt_vehicles.add.xml